# Can three well logs identify sandstone in unseen wells?
Liam Mora — INDE 577

## Investigation contract
This supporting investigation asks whether gamma ray, bulk density and neutron
porosity can distinguish sandstone-labelled intervals from other labelled rock
types in wells excluded from fitting. It complements sonic-log reconstruction
using the same available measurements; it does not predict oil presence.
A petrophysicist could use a useful classifier to prioritize intervals for review.
Here we assess an initial screening candidate, not an autonomous geological decision.

One observation is a depth interval. The positive class is the FORCE sandstone
label (30000); mixed sandstone/shale (65030) and all other valid labels are negative.
This label definition does not imply that mixed rock contains no sandstone.
The primary decision threshold is fixed at 0.5. All results below are development
validation evidence; the 20 reserved final-test wells are not scored.

## Data provenance and evaluation design
The FORCE 2020 lithology competition data are available from the
[original repository](https://github.com/bolgebrygg/Force-2020-Machine-Learning-competition)
and [archived record](https://zenodo.org/records/4351156), with source attribution
under the Norwegian Licence for Open Government Data (NLOD 2.0).
The local archive was verified on November 19, 2026; its SHA256 is recorded in
`configs/reserved_wells.json`. Acquisition and coverage are documented in
`docs/FORCE_DATA_CHECK.md`. Place the audited archive at `data/local/force-train.zip`
before running; it is excluded from Git.

The loader validates schema, duplicate well/depth pairs, numeric sentinels and
label codes. Split assignments precede complete-case filtering. Of 706,823 training
rows, 431,307 have complete classification inputs/labels; of 240,136 validation
rows, 145,345 qualify. Thus the analysis excludes substantial missing-data coverage
and cannot establish performance on incomplete logs.

The fixed split contains 58 training wells and 20 validation wells. Training uses
up to 250 eligible observations per well (seed 577), giving 14,500 rows. Validation
uses every eligible row in its wells. GR is in API units, RHOB in g/cm³ and NPHI
is a porosity fraction. Well ID, depth, DTC and lithology are not predictors.
Only training data fit the scaler and model. Adjacent depths are dependent: randomly
splitting individual rows would leak well-specific structure. Pooled scores weight
longer logged intervals more heavily; per-well scores show a different perspective.

## Mathematical model
For observation $i$, let $x_i$ contain the three measurements and $y_i\in\{0,1\}$
the sandstone label. Standardize each feature using training mean $\mu_j$ and
standard deviation $s_j$: $z_{ij}=(x_{ij}-\mu_j)/s_j$.
With intercept $b$ and weights $w$, the score is $a_i=b+w^Tz_i$ and
$p_i=\sigma(a_i)=1/(1+e^{-a_i})$.

$$L(w,b)=\frac1n\sum_i[\log(1+e^{a_i})-y_i a_i].$$

With design matrix $A=[1,Z]$ and parameter vector $\theta=(b,w)$,
$\nabla L=A^T(p-y)/n$ and $\theta\leftarrow\theta-\eta\nabla L$.
The implementation uses stable sigmoid/softplus calculations, zero initialization,
learning rate 0.2, at most 20,000 updates and gradient-norm tolerance $10^{-7}$.
It has no class weighting or regularization. Predictions are positive when
$p_i\geq0.5$; probabilities are scores whose calibration must be assessed,
not assumed. A linear log-odds boundary may miss nonlinear geological structure.

## Implementation and verification
`BatchLogisticClassifier` in `src/dsm_project/algorithms/student/logistic_regression.py`
implements the model following Lecture 11's logistic-model and decision-policy
approach. `force_logistic.run_experiment` loads data through the project adapter,
constructs the whole-well experiment and returns fitted pipelines and reports.
The notebook imports these functions rather than redefining the estimator.
`tests/test_student_logistic.py` checks finite-difference gradients, extreme scores,
an analytic intercept case, threshold behavior, invalid inputs and reference agreement.
The reference is scikit-learn's unregularized logistic regression on the same scaled
sample. The baseline predicts the training sandstone frequency for every interval.

The next cell runs the fixed experiment, then verifies well separation and reference
agreement. It can take a short time on the first run.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import precision_recall_curve

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "configs/force.toml").is_file())
os.chdir(ROOT)
from dsm_project.data.force import FEATURES
from dsm_project.experiments.force_logistic import run_experiment

result = run_experiment()
report = result["report"]
training, validation = result["training"], result["validation"]
models = result["models"]
reserved = set(json.loads((ROOT / "configs/reserved_wells.json").read_text())
               ["reserved_test_wells"])
assert not set(training.WELL) & set(validation.WELL)
assert not (set(training.WELL) | set(validation.WELL)) & reserved
assert report["threshold"] == 0.5
assert report["converged"]
assert report["max_probability_gap_to_reference"] < 1e-4
print("Source SHA256:", report["source_sha256"])
print("Sample SHA256:", report["sample_ids_sha256"])
display(pd.DataFrame({
    "partition": ["training sample", "development validation"],
    "wells": [training.WELL.nunique(), validation.WELL.nunique()],
    "rows": [len(training), len(validation)],
    "sandstone_fraction": [training.sandstone.mean(), validation.sandstone.mean()],
}))

## Exploration with a purpose
The training sample has about 19.1% sandstone versus 14.4% in validation. Sampling
wells equally and differences between wells can both affect this contrast.
The table and training-only distributions below check input scales and class overlap.
All training observations contribute; no validation-informed outlier removal is used.

In [ ]:
display(training.loc[:, list(FEATURES)].describe().T)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
labels = {"GR": "Gamma ray (API)", "RHOB": "Bulk density (g/cm³)",
          "NPHI": "Neutron porosity (fraction)"}
for ax, feature in zip(axes, FEATURES):
    edges = np.histogram_bin_edges(training[feature], bins=40)
    for target, name, color in [(0, "Other labels", "#406b9e"),
                                 (1, "Sandstone", "#c46b22")]:
        ax.hist(training.loc[training.sandstone == target, feature], bins=edges,
                density=True, histtype="step", linewidth=1.8, label=name, color=color)
    ax.set(xlabel=labels[feature], ylabel="Density")
axes[0].legend()
fig.suptitle("Training distributions by class: overlap limits simple screening")
plt.show()

The density curves normalize each class separately, so they show shape rather
than prevalence. Overlap means no single displayed measurement identifies a class
perfectly. Extreme finite values remain visible; arbitrary clipping would change
the study population. These are associations, not causal effects of rock type.

## Optimization and model comparison
Training loss diagnoses optimization, not generalization. The trusted reference
checks that the custom implementation solves the same objective; agreement does
not establish that the model is geologically adequate.

In [ ]:
fitted = models["batch_logistic"].steps[-1][1]
fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
ax.plot(np.arange(len(fitted.loss_history_)), fitted.loss_history_, color="#406b9e")
ax.set(xlabel="Gradient updates", ylabel="Training mean cross-entropy",
       title="Full-batch optimization converges")
plt.show()
print(f"Updates: {report['iterations']}; gradient norm: {report['gradient_norm']:.3g}")
print(f"Maximum probability difference from reference: {report['max_probability_gap_to_reference']:.3g}")
metrics = pd.DataFrame(report["metrics"]).T.drop(columns="confusion_matrix").astype(float)
display(metrics.round(4))

The custom model converges in 4,016 updates and matches reference probabilities
within approximately 0.0000034. At threshold 0.5, balanced accuracy is 65.3%
versus the baseline's 50%. Precision is 45.6% and recall 38.3%: fewer than half of
flagged intervals are sandstone, and most sandstone intervals are missed.
Accuracy is **84.5%, below the majority baseline's 85.6%**. The baseline wins this
metric by ignoring the minority class. Balanced accuracy averages sensitivity and
specificity; precision measures flag reliability and recall measures coverage.
Average precision (0.370 versus prevalence 0.144) summarizes ranking. Log loss and
Brier score are lower than baseline, supporting improved probability predictions
without proving calibration or practical usefulness.

## Failure analysis: missed intervals and false flags
Rows in the confusion matrix are observed classes and columns are predicted classes.
The precision-recall curve examines score ranking across thresholds; its horizontal
line is the constant-score baseline, equal to validation prevalence.

In [ ]:
y = validation.sandstone.astype(int).to_numpy()
probability = models["batch_logistic"].predict_proba(validation.loc[:, FEATURES])[:, 1]
matrix = np.asarray(report["metrics"]["batch_logistic"]["confusion_matrix"])
precision, recall, _ = precision_recall_curve(y, probability)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].imshow(matrix, cmap="Blues")
for (i, j), count in np.ndenumerate(matrix):
    axes[0].text(j, i, f"{count:,}", ha="center", va="center",
                 color="white" if count > matrix.max()/2 else "black")
axes[0].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Other", "Sandstone"],
            yticklabels=["Other", "Sandstone"], xlabel="Predicted class",
            ylabel="Observed class", title="Confusion matrix at threshold 0.5")
axes[1].plot(recall, precision, label="Custom logistic", color="#406b9e")
axes[1].axhline(y.mean(), linestyle="--", color="#777777", label="Constant-score baseline")
axes[1].scatter(report["metrics"]["batch_logistic"]["recall"],
                report["metrics"]["batch_logistic"]["precision"],
                color="#c46b22", label="Fixed threshold 0.5", zorder=3)
axes[1].set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1),
            title="Ranking supports a precision–recall trade-off")
axes[1].legend()
plt.show()

The fixed policy finds 8,003 sandstone intervals but misses 12,898 and flags
9,564 other intervals. These are correlated depth observations, not independent
discoveries. Precision-recall trade-offs matter if missed intervals and review
workload have different costs; neither cost is specified here.

### Threshold sensitivity, not a new selected policy
The grid below was predefined from 0.1 to 0.9. It describes development behavior;
the primary threshold remains 0.5. Choosing a cutoff after viewing this table would
be tuning and would require independent final evaluation after freezing the choice.

In [ ]:
thresholds = result["thresholds"]
display(thresholds.round(4))
fig, ax = plt.subplots(figsize=(7, 3.8), constrained_layout=True)
for metric in ["balanced_accuracy", "precision", "recall"]:
    ax.plot(thresholds.threshold, thresholds[metric], marker="o", label=metric.replace("_", " "))
ax.axvline(0.5, color="gray", linestyle="--", label="Fixed primary cutoff")
ax.set(xlabel="Decision threshold", ylabel="Validation metric", ylim=(0, 1),
       title="Lower cutoffs trade more review workload for recall")
ax.legend()
plt.show()

At 0.2, recall rises to 77.2% while precision falls to 34.8%; this is a trade-off,
not a free improvement. We retain 0.5 for the initial comparison and make no claim
that the highest balanced accuracy in this grid is an independently validated optimum.

### Between-well variation and uncertainty
Per-well metrics reveal whether pooled results conceal local failures. Balanced
accuracy is shown alongside prevalence and precision because the same sensitivity
can have very different usefulness in a rare-positive well. The well distribution
describes heterogeneity; it is not a confidence interval. We avoid a row-wise
bootstrap because neighboring depth observations are dependent.

In [ ]:
well_metrics = result["per_well"].query("model == 'batch_logistic'").copy()
well_metrics["prevalence"] = well_metrics.positives / well_metrics.rows
well_metrics = well_metrics.sort_values("balanced_accuracy")
display(well_metrics[["well", "rows", "prevalence", "balanced_accuracy", "precision", "recall"]].round(4))
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
ax.scatter(well_metrics.balanced_accuracy, well_metrics.well, color="#406b9e")
ax.axvline(0.5, color="gray", linestyle="--", label="Majority baseline")
ax.set(xlabel="Balanced accuracy at threshold 0.5", ylabel="Validation well",
       xlim=(0, 1), title="Performance varies substantially between unseen wells")
ax.legend()
plt.show()
display(well_metrics[["balanced_accuracy", "precision", "recall"]].agg(["mean", "median", "min", "max"]).round(4))

Balanced accuracy ranges from about 44.7% to 93.6%. Well 31/3-3 performs below
the majority baseline, while 16/7-4 has high balanced accuracy but only about 9.9%
precision. This illustrates why a high balanced score alone cannot justify deploying
a screening policy. These are descriptive observations on one development split.
Repeated whole-well splits and uncertainty estimates based on wells remain future
work; they must respect the reserved final-test boundary.

## Conclusion and production decision
Three common logs provide some sandstone discrimination on these validation wells,
but the initial linear classifier has low recall and substantial false flags. It
improves ranking, balanced accuracy and probability losses over a constant baseline,
while reducing ordinary accuracy. We retain it as an interpretable development
comparison and **do not promote it for autonomous interpretation**.

Complete-case selection, correlated depths, unequal well sizes, class-definition
choices, geological variation and the linear decision boundary limit the evidence.
Training convergence and library agreement verify implementation rather than field
validity. Probabilities have not been calibrated, and no uncertainty interval or
final-test claim is made. Responsible use requires expert review and documented
error costs; this model is not evidence of hydrocarbons or economic value.

Next, compare a shallow regression tree with the sonic linear model for the primary
project question. Later model selection and final evaluation must freeze features,
hyperparameters and any threshold policy before scoring the 20 reserved wells.

## Reproduce
Use the DSM Project kernel and run all cells from the top. The package runner also
writes local diagnostics under `artifacts/force_logistic/`. Run
`uv run --locked pytest tests/test_student_logistic.py tests/test_classification_notebook.py -q`.
The full notebook test executes only when the audited local archive is present;
CI still checks notebook structure without downloading the field dataset.
Per the notebook standard, committed outputs are cleared. An executed local review
copy is kept under `artifacts/force_logistic/02-sandstone-classification.executed.ipynb`.
See `docs/LOGISTIC_REGRESSION.md` for detailed method and validation evidence.